<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_2/notebooks/W6L2_Standardization_and_z_scores__STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W6L2 — Probability, the Normal Distribution, and z-Scores: Was Lidocaine's Effect Unusual?

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**  ·  *In-class activity*

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## Background

Last week you gave **lidocaine** to a model pain neuron and watched it fire far less than an identical
Control neuron. That comparison was clean because the two neurons were identical in every other way.

Real pain neurons are **not** identical. Each one receives a slightly different amount of input and
fires at a slightly different rate. So when a lidocaine-treated neuron fires slowly, a sceptic can ask:
*maybe that neuron was just naturally slow?*

To answer, you need to know **how untreated pain neurons vary**, and then ask:

> **How likely is it that an untreated neuron would fire as slowly as the lidocaine-treated one, just by
> chance?**

That is a question about **probability**, and answering it takes two ideas. First, **probability** and
the **normal distribution**, whose shape is known so exactly that a mean and a standard deviation are
enough to make precise statements. Second, **standardization**: converting a firing rate into a
**z-score** so the probability attached to *any* rate can be read off a single known curve.

## The data

The dataset was **already generated** with NEURON, the simulator you used in W5, and saved as a CSV:

1. **200 untreated pain neurons.** Each neuron's input current was drawn from a normal spread centred on
   **0.30 nA**, which creates the cell-to-cell variability (`Input_current_nA`).
2. **A steady current ran for exactly 1 second**, just like the steady-current step in W5.
3. **The spikes were counted.** Because the window is one second, the spike count **is** the firing rate
   in hertz (`Firing_rate_Hz`).

In Step 5 you'll compare these 200 untreated neurons with **200 lidocaine-treated neurons** recorded under
the same conditions (`lidocaine_firing_rates.csv`, same columns).

### Symbols used in this notebook

| Symbol | Read it as |
|---|---|
| $P$ | probability |
| $N_E$ | number of times the outcome occurred |
| $N_T$ | total number of scores in the sample |
| $X$ | a raw score (here, one neuron's firing rate) |
| $z$ | the z-score — how many SDs a score sits from the mean |
| $\bar{X}$, $s$ | the **sample** mean and standard deviation |
| $\mu$, $\sigma$ | the **population** mean and standard deviation |

Lecture uses $\mu$ and $\sigma$ for a whole population and $\bar{X}$ and $s$ for a sample you actually
measured. The arithmetic is identical. Our 200 neurons are a sample, so this notebook uses $\bar{X}$ and
$s$ throughout.

## What you'll be able to do by the end

1. Calculate a **probability** from a sample using $P = N_E / N_T$
2. Read probability as **area under a curve**, and list the features of a **normal distribution**
3. Use the **68–95–99.7 rule**, and recognise when a distribution is **not** normal
4. Convert a firing rate into a **z-score** and find the probability of a rate being **below** or
   **above** it
5. Compare two groups over the **same range** of firing rates, and decide whether the treated neurons' firing is **unusual** compared with natural variation

## How this notebook works

**Run every cell in order, from the top.** Click a cell and press **Shift + Enter**.

| | What to do |
|---|---|
| **Ordinary cells** | Just run them. You don't need to understand every line of code. |
| **⚙️ Try it** | At most one per step: a value in the code that **you change and re-run**. |
| **💬 Discussion** | At the very end. Answer using the numbers you found along the way. |

If something breaks, first check that you ran **every cell above it**.

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Load the data and look at it

This cell loads the dataset straight from the course GitHub page — nothing to download. `numpy` does the
maths, `pandas` holds the table, `matplotlib` draws the graphs, and `scipy.stats` gives us the
normal-distribution tools that replace the printed z-table.

Before computing anything clever, **look at the distribution**, and note its centre and spread. Almost
everything below is built from those two numbers.

> **⚙️ Try it — change the number of histogram bars.** Set `BINS = 5`, re-run, then `BINS = 50`. Watch
> whether the **mean**, **median** or **SD** printed above the graph change. Set `BINS` back to **15**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_2/data/firing_rates.csv"

data = pd.read_csv(DATA)
rates = data["Firing_rate_Hz"].to_numpy(dtype=float)

mean = np.mean(rates)
sd = np.std(rates, ddof=1)      # ddof=1 because these 200 neurons are a SAMPLE

print("Loaded", len(data), "untreated pain neurons.\n")
print("Mean firing rate:    %.2f Hz" % mean)
print("Median firing rate:  %.2f Hz" % np.median(rates))
print("Standard deviation:  %.2f Hz" % sd)
print("Range:               %.0f to %.0f Hz" % (rates.min(), rates.max()))

BINS = 15      # ⚙️ Try it — number of histogram bars. Try 5, then 50.

plt.figure(figsize=(7, 4))
plt.hist(rates, bins=BINS, color="lightsteelblue", edgecolor="black")
plt.axvline(mean, color="darkred", lw=2, label="mean")
plt.axvline(np.median(rates), color="navy", lw=2, ls="--", label="median")
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Number of neurons")
plt.title("200 untreated pain neurons (%d bins)" % BINS)
plt.legend()
plt.show()

data.head()

## Step 2 — From frequency to probability

**Probability** is the likelihood of a particular event happening. In a sample of scores it is defined
as:

$$ P(\text{event}) = \frac{N_E}{N_T} $$

where $N_E$ is the number of times the outcome occurred and $N_T$ is the total number of scores.
Probability is just counting: if 4 of the 20 pain neurons you recorded fired above 80 Hz,
$P(\text{above 80 Hz}) = 4/20 = 0.2$.

This is the same calculation as the **relative frequency** ($rf = f/N$) from Unit 1. A relative frequency
table *is* a probability distribution — you were computing probabilities all along.

For a **continuous** variable like firing rate, the probability of one exact value isn't useful, so we
ask about a **range**:

$$ P(a < X < b) = \frac{N_E}{N_T} \qquad \text{where } N_E = \text{number of scores between } a \text{ and } b $$

> **⚙️ Try it — change the range.** Run the cell at `60, 70`, then try a wide range (`40, 100`) and a very
> narrow one (`65, 66`). Can $P$ ever fall below 0 or rise above 1?

In [ ]:
low, high = 60, 70        # ⚙️ Try it — the range of firing rates (Hz)

N_E = np.sum((rates >= low) & (rates <= high))   # neurons in the range
N_T = len(rates)                                 # total neurons
P = N_E / N_T

print("N_E (neurons firing between %d and %d Hz) = %d" % (low, high, N_E))
print("N_T (total neurons)                       = %d" % N_T)
print("P = N_E / N_T = %d / %d = %.3f" % (N_E, N_T, P))
print()
print("A randomly chosen untreated neuron has about a %.1f%% chance of firing in that range." % (100 * P))

plt.figure(figsize=(7, 4))
counts, bins, patches = plt.hist(rates, bins=15, density=True,
                                 color="lightsteelblue", edgecolor="black")
for patch, left in zip(patches, bins[:-1]):
    if low <= left < high:
        patch.set_facecolor("mediumseagreen")
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Relative frequency (density)")
plt.title("Shaded bars = neurons between %d and %d Hz" % (low, high))
plt.show()

## Step 3 — The normal distribution and the 68–95–99.7 rule

Many continuous variables — soma diameter, resting membrane potential, and as you're about to see,
firing rate — form a bell-shaped **normal distribution**. It has three defining features:

- **Symmetric** — the two halves mirror each other
- **Unimodal**, with **mean = median = mode** all at the peak
- **Asymptotic** — the tails approach zero but never touch it

Because it is a smooth curve, probability becomes **area under the curve**, and because its shape is
known exactly, the percentage of scores within any distance of the mean is known too:

| Range | Percentage of scores |
|---|---|
| mean ± 1 SD | about **68%** |
| mean ± 2 SD | about **95%** |
| mean ± 3 SD | about **99.7%** |

All of it rests on one assumption: **that the distribution really is normal.** The cell below draws the
normal curve over the data and checks the rule against the neurons we actually have.

> **⚙️ Try it — compare with data that are not normal.** Run with `DATASET = "neurons"`, then set
> `DATASET = "skewed"` (latency to the first spike, a right-skewed measurement) and re-run. Compare the
> "within 1 SD" percentages. Set `DATASET` back to `"neurons"`.

In [ ]:
DATASET = "neurons"      # ⚙️ Try it — "neurons" or "skewed"

if DATASET == "neurons":
    values = rates
    label = "Firing rate (Hz)"
else:
    rng = np.random.default_rng(7)
    values = rng.lognormal(np.log(55), 0.85, 200)     # a strongly right-skewed variable
    label = "Latency to first spike (ms)"

m = np.mean(values)
s = np.std(values, ddof=1)

plt.figure(figsize=(7.5, 4.5))
plt.hist(values, bins=15, density=True, alpha=0.6, color="mediumseagreen",
         edgecolor="black", label="the data")

x = np.linspace(values.min() - 10, values.max() + 10, 300)
plt.plot(x, norm.pdf(x, m, s), color="darkred", lw=2.5, label="normal curve")
plt.axvspan(m - s, m + s, color="red", alpha=0.12, label="mean ± 1 SD")
plt.axvline(m, color="black", lw=2)

plt.xlabel(label)
plt.ylabel("Relative frequency (density)")
plt.title("%s, with a normal curve drawn over it" % DATASET)
plt.legend(fontsize=8)
plt.show()

print("DATASET = %s" % DATASET)
print("Mean = %.2f, median = %.2f, SD = %.2f" % (m, np.median(values), s))
print("(in a perfect normal distribution the mean and median are equal)\n")
for k, expected in [(1, 68), (2, 95), (3, 99.7)]:
    within = 100 * np.mean(np.abs(values - m) <= k * s)
    print("Within %d SD: %.1f%% observed   (normal predicts ~%.1f%%)" % (k, within, expected))

## Step 4 — Standardization: from a score to a probability

The 68–95–99.7 rule only works at **whole** numbers of standard deviations. A **z-score** removes that
limit by expressing a raw score as a distance from the mean, in standard deviations:

$$ z = \frac{X - \bar{X}}{s} $$

A z-score of $+1.5$ means "one and a half standard deviations **above** the mean"; $-0.72$ means "0.72
standard deviations **below**." It changes the ruler, not the data.

Once a score is standardized, `norm.cdf(z)` gives the area under the standard normal curve below it —
the same number a printed z-table gives:

| Question | Code |
|---|---|
| Probability of a score **below** z | `norm.cdf(z)` |
| Probability of a score **above** z | `1 - norm.cdf(z)` |
| Probability **between** two z values | `norm.cdf(z2) - norm.cdf(z1)` |

> **⚙️ Try it — standardize different firing rates.** Run at `test_rate = 80`, then set it to the mean
> (about 66) and then to **50**. Watch the **sign** of z each time. Set it back to **80**.

In [ ]:
# Standardize every neuron in the dataset
z_scores = (rates - mean) / sd

plt.figure(figsize=(11, 3.8))
plt.subplot(1, 2, 1)
plt.hist(rates, bins=15, color="lightsteelblue", edgecolor="black")
plt.axvline(mean, color="darkred", lw=2)
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Number of neurons")
plt.title("Raw scores")
plt.subplot(1, 2, 2)
plt.hist(z_scores, bins=15, color="mediumseagreen", edgecolor="black")
plt.axvline(0, color="darkred", lw=2)
plt.xlabel("z-score (SDs from the mean)")
plt.title("Standardized scores")
plt.tight_layout()
plt.show()


test_rate = 80        # ⚙️ Try it — a firing rate (Hz) to standardize

z = (test_rate - mean) / sd
p_below = norm.cdf(z)
p_above = 1 - p_below

print("A neuron firing at %g Hz:" % test_rate)
print("   z = (%g - %.2f) / %.2f = %+.2f" % (test_rate, mean, sd, z))
print("   It sits %.2f standard deviations %s the mean." % (abs(z), "above" if z > 0 else "below"))
print()
print("   P(firing slower) = %.4f  ->  about %.1f%% of untreated neurons fire slower" % (p_below, 100 * p_below))
print("   P(firing faster) = %.4f  ->  about %.1f%% of untreated neurons fire faster" % (p_above, 100 * p_above))
print()
print("   In our actual 200 neurons, %.1f%% fired faster." % (100 * np.mean(rates > test_rate)))

x = np.linspace(-4, 4, 400)
plt.figure(figsize=(7, 4))
plt.plot(x, norm.pdf(x), color="black")
plt.fill_between(x[x >= z], norm.pdf(x[x >= z]), color="tomato", alpha=0.6,
                 label="P(faster) = %.3f" % p_above)
plt.axvline(z, color="darkred", lw=2)
plt.xlabel("z-score")
plt.ylabel("Probability density")
plt.title("Area under the standard normal curve above z = %+.2f" % z)
plt.legend()
plt.show()

## Step 5 — Back to Lidocaine: compare the two groups

Now the question this whole notebook has been building to. In W5, lidocaine made a pain neuron fire far less than its identical Control. Here we have a **real lidocaine group**: 200 pain neurons, each exposed to lidocaine and given the same kind of 1-second stimulus as the untreated neurons, with the same cell-to-cell variation in input current.

Untreated neurons vary, so could a naturally slow untreated neuron fire as slowly as these? We can ask the **same probability question of both groups**, using $P = N_E / N_T$ from Step 2: *over the same range of firing rates, how likely is a randomly chosen neuron from each group to land in it?*

We then put the lidocaine group **on the untreated ruler** by standardizing its firing rates against the untreated mean and SD, and ask how often an **untreated** neuron fires that slowly just by natural variation.

> **⚙️ Try it — change the range.** Run at `RANGE_LOW, RANGE_HIGH = 60, 70`, then `0, 20`, then something in between such as `20, 45`. Is there any range where **both** groups have a probability above 0? Set it back to **60, 70**.

In [ ]:
# Load the lidocaine group: same columns as the untreated file
LIDO_DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_2/data/lidocaine_firing_rates.csv"
lido = pd.read_csv(LIDO_DATA)
lido_rates = lido["Firing_rate_Hz"].to_numpy(dtype=float)

print("Loaded", len(lido), "lidocaine-treated pain neurons.\n")
print("             untreated    lidocaine")
print("Mean (Hz)    %9.2f    %9.2f" % (mean, np.mean(lido_rates)))
print("Median (Hz)  %9.2f    %9.2f" % (np.median(rates), np.median(lido_rates)))
print("SD (Hz)      %9.2f    %9.2f" % (sd, np.std(lido_rates, ddof=1)))
print("Range (Hz)   %4.0f to %3.0f    %2.0f to %3.0f" % (rates.min(), rates.max(), lido_rates.min(), lido_rates.max()))
print()

# ---- The same probability question for both groups: P = N_E / N_T over the SAME range ----
RANGE_LOW, RANGE_HIGH = 60, 70      # ⚙️ Try it — the range of firing rates (Hz). Try 0, 20 and then 20, 45.

def probability_in_range(x, low, high):
    N_E = np.sum((x >= low) & (x <= high))     # neurons in the range
    N_T = len(x)                               # total neurons
    return N_E, N_T, N_E / N_T

for name, group in [("Untreated", rates), ("Lidocaine", lido_rates)]:
    N_E, N_T, P = probability_in_range(group, RANGE_LOW, RANGE_HIGH)
    print("%-9s  P(%d-%d Hz) = %3d / %d = %.3f   (%.1f%% of neurons)" % (name, RANGE_LOW, RANGE_HIGH, N_E, N_T, P, 100 * P))

# ---- Put the lidocaine group on the untreated ruler ----
def as_percent(p):
    """Write a probability as a percentage, or as '1 in ...' when it is tiny."""
    if p >= 0.001:
        return "%.1f%%" % (100 * p)
    n = 1 / p
    for word, size in [("billion", 1e9), ("million", 1e6), ("thousand", 1e3)]:
        if n >= size:
            return "less than 0.1%%  (about 1 in %.0f %s)" % (n / size, word)

z_lido_mean = (np.mean(lido_rates) - mean) / sd
fastest = lido_rates.max()
z_fastest = (fastest - mean) / sd
p_pred = norm.cdf(z_fastest)

print()
print("The lidocaine group's MEAN sits at z = %+.2f (%.2f SDs %s the untreated mean)."
      % (z_lido_mean, abs(z_lido_mean), "above" if z_lido_mean > 0 else "below"))
print("Its FASTEST neuron (%d Hz) sits at z = %+.2f." % (fastest, z_fastest))
print("   Predicted share of UNTREATED neurons firing that slowly or slower: %s" % as_percent(p_pred))
print("   Among our actual 200 untreated neurons: %d fired that slowly or slower."
      % np.sum(rates <= fastest))

# ---- Plot ----
SHARED_BINS = np.arange(0, 100, 5)

plt.figure(figsize=(7, 4))
plt.hist(rates, bins=SHARED_BINS, color="lightsteelblue", edgecolor="black", alpha=0.85,
         label="untreated")
plt.hist(lido_rates, bins=SHARED_BINS, color="darkorange", edgecolor="black", alpha=0.7,
         label="lidocaine")
plt.axvspan(RANGE_LOW, RANGE_HIGH, color="mediumseagreen", alpha=0.25,
            label="range %d-%d Hz" % (RANGE_LOW, RANGE_HIGH))
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Number of neurons")
plt.title("Untreated vs Lidocaine, same bins")
plt.legend(fontsize=8)
plt.show()

## Putting it all together

| Idea | What it says |
|---|---|
| $P = N_E / N_T$ | Probability is counting: occurrences over total |
| Relative frequency = probability | The formula you used in Unit 1 was a probability all along |
| Normal distribution | Symmetric, unimodal, asymptotic; probability is **area under the curve** |
| 68–95–99.7 rule | A mean and an SD are enough to make precise probability statements, **if** the data are normal |
| $z = \dfrac{X - \bar{X}}{s}$ | Raw score → z → probability, with `norm.cdf(z)` |

This is the logic every inferential test is built on: is a result **ordinary** or **surprising** for the
distribution it came from? A z-score is the first and simplest version of that question.

---

### 💬 Discussion

Keep each answer **short** (a number and one sentence is often enough). Use the default ⚙️ values
unless a question says otherwise.

1. **(Step 1)** Report the mean and SD of the untreated neurons. Did changing `BINS` change either
   number?

2. **(Step 2)** In one sentence, what is the probability that a randomly chosen untreated neuron fires
   between 60 and 70 Hz?

3. **(Step 3)** What percentage of the untreated neurons fell within 1 SD of the mean, and what
   percentage of the skewed data did? Is the normal model a fair description of our neurons?

4. **(Step 4)** For a neuron firing at 80 Hz, what percentage of untreated neurons are **predicted** to
   fire faster, and what percentage **actually** did? When you set `test_rate = 50`, was z positive or
   negative, and what does that sign mean?

5. **(Step 5)** For 60 to 70 Hz, what is P for the untreated group and for the lidocaine group? Then
   try 0 to 20 Hz and report both again. Is there any range where **both** groups have a probability
   above 0? Using the z-score of the lidocaine group's mean, could lidocaine's slow firing just be
   natural variation?

*(Before you submit: set `BINS`, `low, high`, `DATASET`, `test_rate` and `RANGE_LOW, RANGE_HIGH` back to their starting values
and run all.)*

**Your answer:** *(double-click here to type)*

>
>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.